In [1]:
import torch
from scripts.NNModel import NeuralNetwork
import scripts.pyTorchTraining as pyTrain
from os.path import join
import yaml

Welcome to JupyROOT 6.28/00


In [2]:
# Inputs to be read from outside

tagger = "OSKaon"
cut_name = "notSamePV_noOSP"
features_file = "union_PROBNN"
repo = "/work/celani/flavour_tagging/classical-taggers"
seed = 45
decay_type = "Bu2JpsiK"
config_file = "lr0.001_bs128_simple_dm0.0"

In [3]:
# I need the total number of features to create a dummy input
features = pyTrain.get_features(tagger=tagger, yaml_file=features_file, repo_path=repo)
with open(join(repo, f'configs/{config_file}.yaml'), 'r') as file:
    config = yaml.safe_load(file)

Remember to 
pip install onnx
pip install onnxruntime

In [4]:
import torch
model_path = "/work/celani/flavour_tagging/Sara_trainings/model.pth"
model = torch.load(model_path)
model.eval()

dummy_input = torch.randn(1, len(features))
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    # opset_version=17
)

/tmp/ipykernel_766409/2886898180.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path)


In [5]:
# Check
import onnx

# Load the ONNX model
model_onnx = onnx.load("model.onnx")

# Inspect input and output
print("Inputs:")
for input in model_onnx.graph.input:
    print(f"  name: {input.name}")
    print(f"  type: {input.type}")

print("\nOutputs:")
for output in model_onnx.graph.output:
    print(f"  name: {output.name}")
    print(f"  type: {output.type}")

Inputs:
  name: input
  type: tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 17
    }
  }
}


Outputs:
  name: output
  type: tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 1
    }
  }
}

